In [1]:
import holidays
import numpy as np
import pandas as pd
import config

UK_HOLIDAYS = pd.to_datetime(
    list(holidays.country_holidays('UK', years=range(2011, 2026)).keys())
)

raw_feature_name, target_feature_name = zip(*[
    ('Date', 'date'),
    ('Time', 'time'),
    ('Ozone', 'O3'),
    ('Nitric oxide', 'NO'),
    ('Nitrogen dioxide', 'NO2'),
    ('Carbon monoxide', 'CO'),
    ('Modelled Wind Direction', 'wind_dir'),
    ('Modelled Wind Speed', 'wind_speed'),
    ('Modelled Temperature', 'temp'),
    ('PM10 particulate matter (Hourly measured)', 'PM10'),
    ('PM2.5 particulate matter (Hourly measured)', 'PM2.5')
])

is_weekend = lambda date: date.dt.day_name().isin(['Saturday', 'Sunday'])
is_holiday = lambda date, holidays: date.isin(holidays)
is_off_day = lambda date, holidays: is_weekend(date) | is_holiday(date, holidays)


def get_day_category(date, holiday_dates):
    """Calculate day category: 0=workday, 1=weekend, 2=day before weekend"""
    next_date = date + pd.Timedelta(days=1)
    return pd.Categorical(np.select(
        [is_off_day(date, holiday_dates), is_off_day(next_date, holiday_dates)],
        [1, 2],
        default=0
    ))


def add_day_category(df, holiday_dates):
    category = get_day_category(df['date'], holiday_dates)
    return df.assign(day_category=lambda x: category)


def add_sin_cos(df, col_name):
    return df.assign(
        **{f'{col_name}_sin': lambda x, c=col_name: np.sin(np.radians(x[c])).round(4),
           f'{col_name}_cos': lambda x, c=col_name: np.cos(np.radians(x[c])).round(4)}
    )


def apply_min_max(df, exclude=None):
    x = df.select_dtypes(include='number')
    if exclude:
        x = x.drop(columns=exclude, errors='ignore')
    df[x.columns] = (2 * (x - x.min()) / (x.max() - x.min()) - 1).round(4)
    return df


def process(csv_year):
    print(f"Processing {csv_year}")
    return (
        pd.read_csv(
            f"{config.raw_csv}/{csv_year}.csv",
            na_values=['No data'],
            parse_dates=['Date'],
            skiprows=config.rows_to_skip,
            skipfooter=1,
            usecols=raw_feature_name,
            engine='python'
        )
        .rename(columns=dict(zip(raw_feature_name, target_feature_name)))
        .assign(time=lambda df: df['time'].str[:2].astype(int))
        .pipe(add_day_category, UK_HOLIDAYS)
        .pipe(add_sin_cos, 'wind_dir')
        .drop(columns=['wind_dir'])
    )

In [2]:
combined_raw = (
    pd.concat([process(year) for year in range(config.start_year, config.end_year + 1)])
    .sort_values(['date', 'time'])
)
combined_raw.to_csv(config.unscaled_csv, index=False)

Processing 2012
Processing 2013
Processing 2014
Processing 2015
Processing 2016
Processing 2017
Processing 2018
Processing 2019
Processing 2020
Processing 2021
Processing 2022
Processing 2023
Processing 2024
Processing 2025


In [3]:
from models import MinMaxScaler

cleaned = combined_raw.drop(columns=['CO']).dropna()
scaler = MinMaxScaler()
scaled = scaler.fit_transform(cleaned, exclude=['wind_dir_sin', 'wind_dir_cos', 'day_category', 'time'])
scaled.to_csv(config.scaled_csv, index=False)

# Block creation

In [4]:
from blocks import make_blocks
from KNNPredictor import KNNPredictor

INPUT_FEATURES = ['O3', 'NO', 'NO2', 'PM10', 'PM2.5', 'wind_speed', 'temp', 'wind_dir_sin', 'wind_dir_cos']
OUTPUT_FEATURES = ['PM2.5']
input_hours = [6, 7, 8]
forecast_hours = [9, 10, 11, 12, 13]

# Evaluation helpers

In [5]:
def compute_rmse(predictions, actuals):
    return np.sqrt(((predictions - actuals) ** 2).mean(axis=0))


def unscale_value(scaled_value, min_val, max_val):
    return (scaled_value + 1) * (max_val - min_val) / 2 + min_val

# Reporting helpers

In [6]:
def print_hourly_rmse(rmse_values, forecast_hours):
    for hour, r in zip(forecast_hours, rmse_values):
        print(f"  Valanda {hour}: RMSE {r:.4f}")
    print(f"Vidutinis RMSE: {np.mean(rmse_values):.4f}")


def compare_predictions(test_dates, predictions, actuals, forecast_hours, unscale_fn, n_samples=5):
    for i in range(min(n_samples, len(test_dates))):
        print(f"\n{test_dates[i].date()}:")
        for h_idx, hour in enumerate(forecast_hours):
            actual    = unscale_fn(actuals[i, h_idx])
            predicted = unscale_fn(predictions[i, h_idx])
            print(f"  Valanda {hour}: prognozė={predicted:.1f}, realiai={actual:.1f} µg/m³")

In [7]:
workdays = scaled[scaled['day_category'] == 0]
train_df  = workdays[workdays['date'].dt.year < 2025]
test_df   = workdays[workdays['date'].dt.year == 2025]

train_inputs, train_outputs, _          = make_blocks(train_df, INPUT_FEATURES, OUTPUT_FEATURES, input_hours, forecast_hours)
test_inputs,  test_outputs,  test_dates = make_blocks(test_df,  INPUT_FEATURES, OUTPUT_FEATURES, input_hours, forecast_hours)

knn = KNNPredictor(k=5)
knn.fit(train_inputs, train_outputs)
predictions = knn.predict(test_inputs)

rmse = compute_rmse(predictions, test_outputs)
print(f"Train blokų: {len(train_inputs)}, Test blokų: {len(test_inputs)}")
print_hourly_rmse(rmse, forecast_hours)

Train blokų: 1797, Test blokų: 134
  Valanda 9: RMSE 0.0908
  Valanda 10: RMSE 0.0867
  Valanda 11: RMSE 0.0907
  Valanda 12: RMSE 0.1007
  Valanda 13: RMSE 0.0951
Vidutinis RMSE: 0.0928


In [8]:
pm25_min = scaler.min_['PM2.5']
pm25_max = scaler.max_['PM2.5']

def unscale(val):
    return unscale_value(val, pm25_min, pm25_max)

print("PM2.5 RMSE (µg/m³):")
rmse_real = [
    np.sqrt(((unscale(predictions[:, h]) - unscale(test_outputs[:, h])) ** 2).mean())
    for h in range(len(forecast_hours))
]
print_hourly_rmse(rmse_real, forecast_hours)

PM2.5 RMSE (µg/m³):
  Valanda 9: RMSE 6.0190
  Valanda 10: RMSE 5.7486
  Valanda 11: RMSE 6.0165
  Valanda 12: RMSE 6.6777
  Valanda 13: RMSE 6.3032
Vidutinis RMSE: 6.1530


In [9]:
print("Pirmos 5 dienos:")
compare_predictions(test_dates, predictions, test_outputs, forecast_hours, unscale)

Pirmos 5 dienos:

2025-01-06:
  Valanda 9: prognozė=7.1, realiai=2.0 µg/m³
  Valanda 10: prognozė=7.0, realiai=2.0 µg/m³
  Valanda 11: prognozė=6.7, realiai=-1.0 µg/m³
  Valanda 12: prognozė=6.3, realiai=3.0 µg/m³
  Valanda 13: prognozė=8.1, realiai=3.0 µg/m³

2025-01-07:
  Valanda 9: prognozė=8.4, realiai=6.0 µg/m³
  Valanda 10: prognozė=9.2, realiai=5.0 µg/m³
  Valanda 11: prognozė=9.0, realiai=2.0 µg/m³
  Valanda 12: prognozė=10.2, realiai=4.0 µg/m³
  Valanda 13: prognozė=12.7, realiai=5.0 µg/m³

2025-01-08:
  Valanda 9: prognozė=13.6, realiai=8.0 µg/m³
  Valanda 10: prognozė=12.3, realiai=8.0 µg/m³
  Valanda 11: prognozė=17.6, realiai=11.0 µg/m³
  Valanda 12: prognozė=16.7, realiai=11.0 µg/m³
  Valanda 13: prognozė=15.9, realiai=10.0 µg/m³

2025-01-09:
  Valanda 9: prognozė=9.8, realiai=12.0 µg/m³
  Valanda 10: prognozė=9.1, realiai=15.0 µg/m³
  Valanda 11: prognozė=8.7, realiai=18.0 µg/m³
  Valanda 12: prognozė=9.0, realiai=19.0 µg/m³
  Valanda 13: prognozė=8.6, realiai=13.0 µg/m³

In [10]:
print(f"{'Val.':<6} {'Reali (vid.)':<16} {'KNN (vid.)':<14} {'Naive':<14} {'KNN RMSE':<12} {'Naive RMSE':<12} Geriau")
print("-" * 82)
for h_idx, hour in enumerate(forecast_hours):
    actual    = unscale(test_outputs[:, h_idx])
    knn_pred  = unscale(predictions[:, h_idx])
    naive_val = unscale(train_outputs[:, h_idx]).mean()

    knn_rmse   = np.sqrt(((knn_pred  - actual) ** 2).mean())
    naive_rmse = np.sqrt(((naive_val - actual) ** 2).mean())
    winner     = "KNN" if knn_rmse < naive_rmse else "Naive"

    print(f"{hour:<6} {actual.mean():<16.1f} {knn_pred.mean():<14.1f} {naive_val:<14.1f} {knn_rmse:<12.2f} {naive_rmse:<12.2f} {winner}")

Val.   Reali (vid.)     KNN (vid.)     Naive          KNN RMSE     Naive RMSE   Geriau
----------------------------------------------------------------------------------
9      12.8             12.4           16.8           6.02         11.90        KNN
10     12.4             12.7           16.7           5.75         11.27        KNN
11     11.5             12.5           16.4           6.02         10.91        KNN
12     11.4             12.5           16.1           6.68         10.23        KNN
13     10.7             12.2           15.9           6.30         9.69         KNN


In [11]:
print("Dienos su aukštu PM2.5 9 val. (>20 µg/m³):")
for i, date in enumerate(test_dates):
    if unscale(test_outputs[i, 0]) > 20:
        print(f"\n{date.date()}:")
        for h_idx, hour in enumerate(forecast_hours):
            actual    = unscale(test_outputs[i, h_idx])
            predicted = unscale(predictions[i, h_idx])
            print(f"  Valanda {hour}: prognozė={predicted:.1f}, realiai={actual:.1f} µg/m³")

Dienos su aukštu PM2.5 9 val. (>20 µg/m³):

2025-01-22:
  Valanda 9: prognozė=14.1, realiai=29.0 µg/m³
  Valanda 10: prognozė=15.0, realiai=27.0 µg/m³
  Valanda 11: prognozė=15.7, realiai=17.0 µg/m³
  Valanda 12: prognozė=13.9, realiai=24.0 µg/m³
  Valanda 13: prognozė=11.0, realiai=27.0 µg/m³

2025-02-06:
  Valanda 9: prognozė=12.3, realiai=21.0 µg/m³
  Valanda 10: prognozė=16.5, realiai=19.0 µg/m³
  Valanda 11: prognozė=14.4, realiai=15.0 µg/m³
  Valanda 12: prognozė=12.8, realiai=10.0 µg/m³
  Valanda 13: prognozė=14.5, realiai=5.0 µg/m³

2025-02-10:
  Valanda 9: prognozė=32.6, realiai=36.0 µg/m³
  Valanda 10: prognozė=33.9, realiai=37.0 µg/m³
  Valanda 11: prognozė=29.7, realiai=33.0 µg/m³
  Valanda 12: prognozė=28.8, realiai=35.0 µg/m³
  Valanda 13: prognozė=28.6, realiai=30.0 µg/m³

2025-02-11:
  Valanda 9: prognozė=23.3, realiai=29.0 µg/m³
  Valanda 10: prognozė=27.4, realiai=33.0 µg/m³
  Valanda 11: prognozė=29.8, realiai=38.0 µg/m³
  Valanda 12: prognozė=29.7, realiai=49.0 µg/m

In [17]:
date = '2025-03-11'
day_category = get_day_category(pd.Series([pd.Timestamp(date)]), UK_HOLIDAYS)[0]

hist_df = scaled[
    (scaled['date'] < date) &
    (scaled['day_category'] == day_category)
]
hist_inputs, hist_outputs, _ = make_blocks(hist_df, INPUT_FEATURES, OUTPUT_FEATURES, input_hours, forecast_hours)

day_input = (
    scaled[(scaled['date'] == date) & (scaled['time'].isin(input_hours))]
    .sort_values('time')[INPUT_FEATURES]
    .values.flatten()
)

prediction = KNNPredictor(k=5).fit(hist_inputs, hist_outputs).predict([day_input])[0]

actual = (
    scaled[(scaled['date'] == date) & (scaled['time'].isin(forecast_hours))]
    .sort_values('time')['PM2.5']
    .values
)

print(f"{date}:")
for h_idx, hour in enumerate(forecast_hours):
    print(f"  Valanda {hour}: prognozė={unscale(prediction[h_idx]):.1f}, realiai={unscale(actual[h_idx]):.1f} µg/m³")

2025-03-11:
  Valanda 9: prognozė=12.6, realiai=9.0 µg/m³
  Valanda 10: prognozė=10.8, realiai=8.0 µg/m³
  Valanda 11: prognozė=16.8, realiai=10.0 µg/m³
  Valanda 12: prognozė=9.6, realiai=7.0 µg/m³
  Valanda 13: prognozė=9.8, realiai=9.0 µg/m³
